In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [9]:
import os
import torch
os.environ["WANDB_DISABLED"] = "true"
import numpy as np
import tensorflow as tf
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset, concatenate_datasets
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 1000)



file_path = '/content/drive/MyDrive/tawfeeq/RE_dataset.json'




df = pd.read_json(file_path)





df.fillna('', inplace=True)

display(len(df))
display(df[:4])



df = df[df['text'] != '']

classes = set(df['label'].values)
display(classes)

# return

df['label'] = df['label'].astype('category')
df['label'] = df['label'].cat.codes



df = df[['text', 'label']]


classes_num = len(classes)
display(classes_num)
display(len(df))

display(df[:4])

ds = Dataset.from_pandas(df)

ds = ds.train_test_split(test_size=0.2, seed=42)
display(ds)

max_sequence_length = 128


models = [
    # 'microsoft/deberta-v3-base',
    # 'FacebookAI/roberta-base',
    'bert-base-cased',
]

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

class BertLSTMModel(nn.Module):
    def __init__(self, model_name, num_labels, hidden_dim=128, num_layers=2):
        super(BertLSTMModel, self).__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        self.lstm = nn.LSTM(input_size=self.bert.config.hidden_size,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, labels=None):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_outputs.last_hidden_state
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = lstm_output[:, -1, :]  # Take the output of the last time step
        logits = self.classifier(lstm_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
        )



for model_name in models:
    for i in range(1):
        print(f'{model_name}, try:{i}')

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = BertLSTMModel(model_name=model_name, num_labels=classes_num).to('cuda')

        dataset_train = ds['train']
        dataset_validation = ds['test']

        def preprocess_function(examples):
            return tokenizer(examples['text'], truncation=True, padding="max_length",
                             max_length=max_sequence_length, add_special_tokens=True)

        dataset_train = dataset_train.map(preprocess_function, batched=True)
        dataset_validation = dataset_validation.map(preprocess_function, batched=True)

        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)
            acc = accuracy_score(labels, predictions)
            f1 = f1_score(labels, predictions, average='macro')
            with open(log_file, 'a') as f:
                f.write(f'{model_name},{acc},{f1}\n')
            return {'accuracy': acc, 'f1_score': f1}

        epochs = 25
        save_steps = 10000
        batch_size = 32

        training_args = TrainingArguments(
            # output_dir='bert/',
            # overwrite_output_dir=True,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            save_steps=save_steps,
            save_total_limit=1,
            fp16=True,
            learning_rate=5e-5,
            logging_steps=50,
            # evaluation_strategy='steps',
            eval_steps=50
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset_train,
            eval_dataset=dataset_validation,
            compute_metrics=compute_metrics
        )

        trainer.train()






4983

,label,text
0,0,Yoggie shall coordinate on future enhancement and features with our organization.
1,0,Within-page links should be clearly distinguishable from other links that lead to a different page. EX. Within-page links are shown with dashed rather than solid underlines
2,1,With the information provided by administrator user can directly contact with system or he can contact during their chat.
3,1,"Whilst on-going, a multi-drivers indication shall be displayed permanently at all Cab radios."


{np.int64(0), np.int64(1)}

2

4983

...

bert-base-cased, try:0


Map:   0%|          | 0/3986 [00:00<?, ? examples/s]

Map:   0%|          | 0/997 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
50,0.573500
100,0.426800
150,0.353200
200,0.306300
250,0.280200
300,0.204600
350,0.198100
400,0.158300
450,0.140600
500,0.197100


In [12]:
# ---- Predict on validation (test) split ----
pred = trainer.predict(dataset_validation)
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=-1)

# Confidence of the predicted class
probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()
conf = probs[np.arange(len(y_pred)), y_pred]

# ---- Label mapping (manual, since custom model has no .config) ----
id2label = {0: "NF", 1: "F"}
label_names = ["NF", "F"]

# ---- Misclassifications ----
mis_idx = np.where(y_pred != y_true)[0]

if "id" not in val_raw.column_names:
    val_raw = val_raw.add_column("id", list(range(len(val_raw))))
rows = []
for i in mis_idx:
    idx = int(i)  # fix: convert numpy.int64 → python int
    rows.append({
        "id": val_raw[idx]["id"],
        "text": val_raw[idx]["text"],
        "true_label_id": int(y_true[idx]),
        "true_label": id2label[int(y_true[idx])],
        "pred_label_id": int(y_pred[idx]),
        "pred_label": id2label[int(y_pred[idx])],
        "pred_confidence": float(conf[idx]),
    })

df_mis = pd.DataFrame(rows)

df_mis.to_json("misclassified_test.json", orient="records", force_ascii=False, indent=2)

print("done finding misclassified")


done finding misclassified


In [16]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# ---- Generate predictions with fine-tuned model ----
pred = trainer.predict(dataset_validation)  # uses model + dataset_validation
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=-1)

# ---- Confusion Matrix ----
custom_labels = ["NF", "F"]  # mapping: 0 -> NF, 1 -> F

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1], normalize="true")

# Save normalized confusion matrix as JSON (percentages)
cm_norm_percent = (cm_norm * 100).round(2).tolist()
cm_norm_dict = {
    "labels": custom_labels,
    "matrix_percent": cm_norm_percent
}
with open("confusion_matrix_percent.json", "w", encoding="utf-8") as f:
    json.dump(cm_norm_dict, f, ensure_ascii=False, indent=2)

# Print classification report
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=custom_labels, digits=4))

# ---- Plot and save confusion matrix (percentages) ----
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=custom_labels)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Confusion Matrix (%)")
plt.tight_layout()
plt.savefig("confusion_matrix_percent.png", dpi=200)
plt.close()


Classification report:
              precision    recall  f1-score   support

          NF     0.8510    0.8365    0.8437       471
           F     0.8558    0.8688    0.8623       526

    accuracy                         0.8536       997
   macro avg     0.8534    0.8527    0.8530       997
weighted avg     0.8535    0.8536    0.8535       997

